# Exercise B — Debug a Climate Script
**LEAP Climate Data Science Bootcamp · Summer 2026 - Ayon Roy**

This notebook is **deliberately buggy**. There are **three intentional bugs** in the
analysis below — they're realistic mistakes that show up in actual climate code.
Your job is to find and fix each one **with help from an AI assistant**.

## The setup

Imagine you're a graduate student who just inherited this notebook from a
lab-mate who's now at a postdoc somewhere. You're supposed to use it to
compute temperature anomalies over a small Amazon-basin grid, but the numbers
look weird. Time to figure out why.

## What you'll do

1. Run the whole notebook top to bottom. Look at the printed numbers and the figure.
2. Find a cell whose output looks suspicious.
3. Paste that cell **plus** the output **plus** what you expected into your AI tool.
4. Apply the fix. Re-run. Verify the new output makes physical sense.
5. Write a one-sentence note in the notebook about each fix.

## Recommended prompt template

> *I'm analyzing ERA5 monthly 2 m temperature for a small lat/lon box, 1979–2023.
> Here is my code [paste cell] and the output [paste numbers or traceback].
> The expected behavior is [describe in one sentence]. What's the bug, and show me
> the corrected cell with comments explaining the change.*

There are **3 bugs**. They are flagged inline with `# BUG #N` comments — but
**don't peek** at the bug labels until after you've tried to find each one yourself.
Treat the labels as a hint of last resort.

## The data

`sample_era5.nc` is a synthetic ERA5-like file with a 2 m temperature field on
a coarse Amazon-basin grid, monthly, 1979–2023. The data is in **Kelvin**
(ERA5's native unit). There's a built-in warming trend of about 0.02 K/year so
you can sanity-check your anomaly calculation.

## 1. Setup

In [ ]:
# If you're on Colab, uncomment the next line to install pinned deps
# !pip install -q xarray netCDF4 matplotlib numpy pandas

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('xarray', xr.__version__)
print('numpy ', np.__version__)

## 2. Load the data

If running locally, make sure `sample_era5.nc` is in the same folder as this notebook.
On Colab, upload it via the file panel (left sidebar).

In [ ]:
ds = xr.open_dataset('sample_era5.nc')
ds

In [ ]:
# Quick sanity check
print('Variable:', list(ds.data_vars))
print('Units of t2m:', ds.t2m.attrs.get('units', 'NOT SET'))
print('Time range:', ds.time.values[0], '→', ds.time.values[-1])
print('Lat range :', float(ds.lat.min()), '→', float(ds.lat.max()))
print('Lon range :', float(ds.lon.min()), '→', float(ds.lon.max()))
print('Mean t2m  :', float(ds.t2m.mean()), 'K')

## 3. Compute a regional time-series

We want a **single time series** that represents the regional-average temperature
at each time step. So we need to reduce over `lat` and `lon` but **keep `time`**.
Let's get the area-mean (we'll ignore cos(lat) weighting for now, the box is small).

> 🔎 *Look carefully at the next cell.*

In [ ]:
# Compute a regional-mean temperature time series
# BUG #1: This reduces over ALL dimensions, including time, instead of just lat/lon.
#         The shape after this line should be (time,) — but it isn't.
regional_mean = ds.t2m.mean()

print('Shape of regional_mean:', regional_mean.shape)
print('regional_mean :', float(regional_mean))

**Stop and think.** What shape did you expect `regional_mean` to have? Did you get it?
If not, that's bug #1. Try fixing it before moving on, then re-run.

*(If you've fixed it, `regional_mean.shape` should be `(540,)` — one value per month.)*

## 4. Compute climatology and anomalies

Now define a **reference climatology** and subtract it to get anomalies.

The **WMO standard** climatological reference period for climate-change monitoring
is **1991–2020** (it gets updated every 10 years). Older papers used 1981–2010 or
1961–1990. Whatever you pick, be **consistent** and **document it**.

> 🔎 *Read the next cell carefully.*

In [ ]:
# Define climatology baseline and compute anomalies
# BUG #2: The climatology window is wrong. The variable is named 'wmo_climatology'
#         but the slice uses 1979-2023 — that's the WHOLE record, not the WMO reference.
#         The right period is 1991-2020. As written, the 'anomaly' is dominated by the
#         long-term warming trend instead of being centered on a reference period.
wmo_climatology = regional_mean.sel(time=slice('1979', '2023')).mean('time')

anomaly = regional_mean - wmo_climatology

print('Climatology value:', float(wmo_climatology), 'K')
print('Anomaly mean     :', float(anomaly.mean()), 'K')
print('Anomaly std      :', float(anomaly.std()), 'K')

**Stop and think.** With a real WMO 1991–2020 baseline and a warming trend in the
data, you would expect the early years to show **negative** anomalies and the recent
years to show **positive** anomalies, with the mean near zero **over the baseline period**.
Does what you got look like that? If the mean over the whole record is suspiciously
close to zero, your baseline is probably the whole record, not 1991–2020. That's bug #2.

## 5. Convert to Celsius for plotting

ERA5 temperatures are in **Kelvin**. To plot in Celsius we subtract **273.15**.

But wait — these are **anomalies**, not absolute temperatures. Anomalies are
**differences in K** (which equal **differences in °C**). So you should **not**
subtract 273.15 from an anomaly.

> 🔎 *Read the next cell carefully.*

In [ ]:
# Convert to Celsius for the plot
# BUG #3: This subtracts 273.15 from an ANOMALY. An anomaly is already a difference
#         in K (= a difference in °C). Subtracting 273.15 turns small anomalies (±1 K)
#         into nonsensical values (~-273 °C). Should ONLY be applied to absolute temps.
anomaly_celsius = anomaly - 273.15

print('Anomaly in (wrongly-converted) °C — first 5 values:')
print(anomaly_celsius.values[:5])

**Stop and think.** Temperature anomalies in the modern climate record are typically
**±1 to ±3 K**. If your printed numbers are around **−273**, you've subtracted 273.15
from something that didn't need it. That's bug #3.

## 6. Plot the time series

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(anomaly_celsius.time, anomaly_celsius.values,
        color='#0D9488', lw=1.0, label='Monthly anomaly')

# Add a 12-month rolling mean for readability
rolling = anomaly_celsius.rolling(time=12, center=True).mean()
ax.plot(rolling.time, rolling.values,
        color='#0E2A47', lw=2.0, label='12-month rolling mean')

ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('Year')
ax.set_ylabel('Temperature anomaly (°C)')
ax.set_title('Regional-mean temperature anomaly\n(synthetic ERA5-style sample)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. What a correct version should look like

After you've fixed all three bugs, your final plot should show:

- Anomalies on the order of **±1 to ±3 °C** (not ±273!).
- A **rising trend** over time — early years mostly negative, recent years mostly positive.
- The **mean over 1991–2020** should be close to zero (that's the baseline period).

If those three things hold, you've got it.

## 8. Your write-up

In the cell below, write 1 sentence per bug describing:

1. **What the bug was.**
2. **How you prompted the AI to find or fix it.**
3. **What you had to verify yourself** that the AI couldn't tell you (e.g. did you trust the WMO baseline citation, or did you look it up?).

This is the **point** of the exercise — not the bugs themselves, but the *meta-skill*
of using AI productively while still doing the verification.

In [ ]:
# YOUR WRITE-UP
# Bug 1: ...
# Bug 2: ...
# Bug 3: ...


## Stretch goals (if you finish early)

1. Use `groupby('time.month')` to compute a **monthly climatology** (one value per
   calendar month, averaged over 1991–2020) and subtract that to get **seasonal anomalies**.
   Did it change the picture?
2. Add **cos(lat) area weighting** to the regional mean. Does it matter at this latitude?
3. Compute the **linear trend** (per decade) using `xarray.polyfit` and ask your AI to
   verify whether the trend is statistically significant.